# AI-powered Research Paper Discovery Assistant
Five domains: **NLP, Healthcare AI, Cybersecurity, Computer Vision, Robotics**.

**Colab:** Runtime → Change runtime type → T4 GPU, then run top to bottom.
Code cell 1 fetches the latest public GitHub commit; cell 2 installs packages; cell 3 contains all adjustable settings.
Two classifiers × four pretrained embeddings = **eight embedding systems**, plus a BoW baseline and zero/few-shot Qwen3 via Ollama.
No API key. Model downloads happen in Colab, not on the local PC. Results are computed when you run; none are pre-filled.


In [ ]:
# 1. Always fetch the latest public code. Rerun after pushing a code change.
import sys, subprocess, tempfile, time
from pathlib import Path
print('START: clone latest public main commit', flush=True)
REPO_URL = 'https://github.com/saltypal/Text_Analytics_Lab_Evaluation.git'
ROOT = Path(tempfile.mkdtemp(prefix='research_lab_checkout_'))
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(ROOT)], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
for name in list(sys.modules):
    if name == 'research_assistant' or name.startswith('research_assistant.'):
        del sys.modules[name]
sys.path.insert(0, str(ROOT))
print('DONE: source directory:', ROOT, 'commit:', COMMIT)

In [ ]:
# 2. All required packages and the Ollama runtime (Colab/Linux only).
print('START: install packages in the active notebook kernel', flush=True)
%pip install -q -r {ROOT / 'requirements.txt'}
import shutil, platform, urllib.request
if shutil.which('ollama') is None:
    if platform.system() != 'Linux':
        raise RuntimeError('Run this installation cell in Google Colab, not Windows.')
    if shutil.which('zstd') is None:
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
    installer = Path(tempfile.gettempdir()) / 'research_lab_install_ollama.sh'
    urllib.request.urlretrieve('https://ollama.com/install.sh', installer)
    subprocess.run(['sh', str(installer)], check=True)
print('DONE: packages and Ollama installed. Restart only if Colab asks, then rerun from cell 1.')

## Configuration — edit only this cell
These settings control the imported modules. Choose them before evaluating; do not tune them on test results. Changing module code on GitHub requires rerunning code cell 1, then later cells. Changing this notebook config only requires rerunning from this cell.

In [ ]:
# 3. ALL adjustable experiment/model settings.
print('START: configuration')
CONFIG = {
    'dataset': 'Q4_Research_Paper_Dataset_1000.csv',
    'seed': 42,
    'test_size': 0.20,
    'preview_papers': 3,
    'bow_top_terms': 20,
    'ngram_orders': [1, 2, 3],
    'ngram_phrases': ['natural language processing', 'patient data', 'robot navigation', 'malware detection'],
    'generated_words': 40,
    'embedding_vocab_limit': 50000,  # First N published pretrained entries; increase for better coverage.
    'static_embeddings': ['Word2Vec', 'GloVe', 'FastText'],
    'bert_model': 'google-bert/bert-base-uncased',
    'bert_batch_size': 16,  # Reduce to 8 if GPU memory is insufficient.
    'bert_max_length': 256,
    'logistic_c': 1.0,
    'svm_c': 1.0,
    'classifier_max_iter': 3000,
    'llm_model': 'qwen3:0.6b',
    'ollama_url': 'http://127.0.0.1:11434',
    'llm_think': False,
    'llm_temperature': 0.0,
    'llm_context': 8192,
    'llm_max_tokens': 128,
    'llm_timeout': 180,
    'shots_per_class': 1,  # Five demonstrations total; only training papers.
    'query': 'patient diagnosis using machine learning',
    'domain_filter': None,  # e.g. 'Healthcare AI'
    'discovery_top_k': 5,
}
print(CONFIG)
print('DONE: config ready. LLM evaluation always uses exactly 10 papers (two per class).')

## Setup, environment and evidence
Each run writes under `experiments/lab_evaluation/runs/<run-id>/`. Colab storage is temporary: download the final ZIP and save the executed notebook using File → Download → .ipynb. Logging starts after setup; pip/native process output and rich displays are preserved by saving the executed notebook.

In [ ]:
print('START: environment setup', flush=True)
import os, json, hashlib, importlib.metadata, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown
from research_assistant.preprocessing import prepare_data, preprocess_steps, bag_of_words, tokens, LABELS
from research_assistant.classification import train_classifiers, metrics, confusion_plots, misclassified
random.seed(CONFIG['seed']); np.random.seed(CONFIG['seed']); torch.manual_seed(CONFIG['seed'])
BASE = Path('/content') if Path('/content').exists() else ROOT
OUT = BASE / 'experiments' / 'lab_evaluation' / 'runs' / (time.strftime('%Y%m%d-%H%M%S')+'-'+str(time.time_ns())[-6:])
OUT.mkdir(parents=True, exist_ok=True)
CACHE = BASE / 'research_lab_cache'; CACHE.mkdir(exist_ok=True)
ENV = {'repository': str(ROOT), 'commit': COMMIT, 'python': sys.version, 'interpreter': sys.executable,
       'packages': {p: importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','nltk','torch','transformers']},
       'device': 'cuda' if torch.cuda.is_available() else 'cpu',
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       'target': 'Google Colab managed Python + requirements.txt in active kernel'}
(OUT/'environment.json').write_text(json.dumps(ENV, indent=2))
(OUT/'config.json').write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(ENV, indent=2)); print('Outputs:', OUT)
# Incremental Python stdout/stderr and cell-status logs. Save .ipynb separately for rich outputs.
ip = get_ipython()
if hasattr(ip, '_lab_hooks'):
    old_pre, old_post = ip._lab_hooks
    ip.events.unregister('pre_run_cell', old_pre); ip.events.unregister('post_run_cell', old_post)
for stream in ['stdout', 'stderr']:
    old = getattr(sys, stream)
    if hasattr(old, '_lab_original'): setattr(sys, stream, old._lab_original)
class Tee:
    def __init__(self, original, path): self._lab_original, self.path = original, path
    def write(self, value):
        self._lab_original.write(value)
        with self.path.open('a', encoding='utf-8') as f: f.write(value)
        return len(value)
    def flush(self): self._lab_original.flush()
    def __getattr__(self, key): return getattr(self._lab_original, key)
state = {}
def before_cell(info):
    state.update(start=time.perf_counter(), cell=hashlib.sha256(info.raw_cell.encode()).hexdigest()[:12])
    print('START cell', state['cell'], flush=True)
def after_cell(result):
    record = {'cell': state.get('cell'), 'execution_count': result.execution_count,
              'elapsed_seconds': round(time.perf_counter()-state.get('start', time.perf_counter()), 3),
              'error': str(result.error_in_exec or result.error_before_exec or '')}
    with (OUT/'cell_status.jsonl').open('a') as f: f.write(json.dumps(record)+'\n')
    print('DONE cell', record, flush=True)
sys.stdout = Tee(sys.stdout, OUT/'stdout.log'); sys.stderr = Tee(sys.stderr, OUT/'stderr.log')
ip.events.register('pre_run_cell', before_cell); ip.events.register('post_run_cell', after_cell)
ip._lab_hooks = (before_cell, after_cell)
RESULTS, PREDICTIONS, MODELS = [], {}, {}
def fit_pair(name, x_train, x_test):
    rows, preds, models = train_classifiers(name, x_train, x_test, TRAIN, TEST,
        seed=CONFIG['seed'], logistic_c=CONFIG['logistic_c'], svm_c=CONFIG['svm_c'], max_iter=CONFIG['classifier_max_iter'])
    RESULTS[:] = [r for r in RESULTS if not r['System'].startswith(name+' +')]+rows
    PREDICTIONS.update(preds); MODELS.update(models)
    pd.DataFrame(RESULTS).to_csv(OUT/'full_test_results.csv', index=False)
    display(pd.DataFrame(rows).round(4))
pd.set_option('display.max_colwidth', 120)
print('DONE: evidence folder writable, logging enabled.')

## Load, deduplicate and split
Join title + abstract only. Remove exact repeated texts before the stratified 80/20 split. Labels never enter the features. Similar templates may remain; results describe this dataset, not arbitrary real-world papers.

In [ ]:
print('START: dataset and split')
DATA = ROOT / CONFIG['dataset']
DF, TRAIN, TEST = prepare_data(DATA, seed=CONFIG['seed'], test_size=CONFIG['test_size'])
ENV['dataset_sha256'] = hashlib.sha256(DATA.read_bytes()).hexdigest()
(OUT/'environment.json').write_text(json.dumps(ENV, indent=2))
TRAIN[['paper_id','research_domain']].to_csv(OUT/'train_split.csv', index=False)
TEST[['paper_id','research_domain']].to_csv(OUT/'test_split.csv', index=False)
display(DF[['paper_id','title','research_domain']].head())
display(pd.DataFrame({'All': DF.research_domain.value_counts(), 'Train': TRAIN.research_domain.value_counts(), 'Test': TEST.research_domain.value_counts()}))
print('DONE: split manifests saved, exact train/test text overlap checked.')

## Preprocessing: every stage
Regex tokenization → case folding → English stopword removal → Porter stemming. No NLTK corpus download needed. Stems are used for BoW; natural tokens/text are used for pretrained embeddings so vocabulary and context are preserved.

In [ ]:
print('START: preprocessing demonstration')
examples = []
for row in TRAIN.head(CONFIG['preview_papers']).itertuples():
    steps = preprocess_steps(row.text); examples.append(steps)
    display(Markdown('### '+row.paper_id+': '+row.title))
    display(pd.DataFrame([{'Stage': k, 'Output': ' | '.join(v) if isinstance(v,list) else v} for k,v in steps.items()]))
pd.DataFrame(examples).to_csv(OUT/'preprocessing_examples.csv', index=False)
print('DONE: examples saved.')

## Bag of Words
Fit vocabulary on training data only. Matrix entries count stemmed words; show five papers and the most frequent training terms.

In [ ]:
print('START: Bag of Words')
BOW, BOW_TRAIN, BOW_TEST = bag_of_words(TRAIN, TEST)
terms = BOW.get_feature_names_out()
counts = np.asarray(BOW_TRAIN.sum(axis=0)).ravel()
top = counts.argsort()[-CONFIG['bow_top_terms']:][::-1]
display(pd.DataFrame(BOW_TRAIN[:5,top].toarray(), columns=terms[top], index=TRAIN.paper_id.head(5)))
table = pd.DataFrame({'Term': terms[top], 'Train count': counts[top]})
display(table); table.to_csv(OUT/'bow_top_terms.csv', index=False)
print('DONE:', BOW_TRAIN.shape, BOW_TEST.shape)

## Unigram, bigram and trigram language models
Use training text tokens with stopwords retained for readable samples. Laplace-smoothed probabilities: `P(w|h)=(count(h,w)+1)/(count(h)+vocabulary_size)`. Sequence probability includes start context, excludes end token. Generation samples observed counts (unsmoothed empirical model).

In [ ]:
from research_assistant.ngrams import NGramLM
print('START: n-gram training')
LMS = {n: NGramLM(n).fit([tokens(t) for t in TRAIN.text]) for n in CONFIG['ngram_orders']}
rows = []
for n, lm in LMS.items():
    for phrase in CONFIG['ngram_phrases']:
        words = tokens(phrase); probability, logp = lm.sequence_probability(words)
        rows.append({'Order':n, 'Sequence':phrase, 'Probability':probability, 'Log probability':logp,
                     'P(last|context)':lm.probability(words[-1], words[:-1])})
    print(f'{n}-gram generated sample:', lm.generate(CONFIG['generated_words'], CONFIG['seed']))
table = pd.DataFrame(rows)
display(table.style.format({'Probability':'{:.8e}', 'Log probability':'{:.3f}', 'P(last|context)':'{:.6f}'}))
table.to_csv(OUT/'ngram_probabilities.csv', index=False)
print('DONE: three models, selected probabilities and samples.')

## BoW baseline: Logistic Regression and Linear SVM
Quick pipeline check before pretrained downloads. All embedding cells below train these same **two** classifiers with the same split and settings.

In [ ]:
print('START: baseline')
fit_pair('Bag of Words', BOW_TRAIN, BOW_TEST)
print('DONE: baseline')

## Pretrained Word2Vec + both classifiers
Average matched pretrained Google News word vectors. Stream the first configured number of published entries to save bandwidth; cache only dataset words. Missing words are skipped. Coverage and zero-vector documents are printed. This is explicitly a limited-vocabulary evaluation.

In [ ]:
from research_assistant.embeddings import load_static_vectors, mean_embeddings, bert_embeddings
print('START: static embedding setup')
VOCABULARY = set(w for text in DF.text for w in tokens(text))
def run_static(name):
    vectors, dim = load_static_vectors(name, VOCABULARY, CACHE, limit=CONFIG['embedding_vocab_limit'])
    x_train = mean_embeddings(TRAIN.text.tolist(), vectors, dim)
    x_test = mean_embeddings(TEST.text.tolist(), vectors, dim)
    np.savez_compressed(OUT/(name.lower()+'_embeddings.npz'), train=x_train, test=x_test)
    fit_pair(name, x_train, x_test)
    print('DONE:', name, 'with BOTH classifiers')
run_static(CONFIG['static_embeddings'][0])

## Pretrained GloVe + both classifiers
50-dimensional Wikipedia/Gigaword embeddings; same pooling, split and classifiers.

In [ ]:
print('START: GloVe')
run_static(CONFIG['static_embeddings'][1])

## Pretrained FastText + both classifiers
Exported 300-dimensional Wikipedia/news word vectors were pretrained with subwords. The export does not include a subword model, so unseen-word synthesis is unavailable. Token coverage makes this limitation visible.

In [ ]:
print('START: FastText')
run_static(CONFIG['static_embeddings'][2])

## Pretrained BERT + both classifiers
Frozen BERT on natural text; attention-mask mean pooling. No BERT fine-tuning. Batch size and maximum token length are configurable above.

In [ ]:
print('START: BERT', flush=True)
matrix = bert_embeddings(TRAIN.text.tolist()+TEST.text.tolist(), CACHE,
    model_name=CONFIG['bert_model'], batch_size=CONFIG['bert_batch_size'], max_length=CONFIG['bert_max_length'])
fit_pair('BERT', matrix[:len(TRAIN)], matrix[len(TRAIN):])
print('DONE: BERT with BOTH classifiers')

## Ollama Qwen3: zero-shot and few-shot
Start local Ollama and download `qwen3:0.6b`. Zero-shot: descriptions only. Few-shot: one training example per domain by default. Evaluate the same 10 held-out papers (two per domain). Thinking disabled; schema constrains JSON to the five labels. Invalid output counts as an error, never as a fabricated fallback label.

In [ ]:
from research_assistant.llm import start_ollama, select_ten, classify_ten
print('START: Ollama setup and LLM evaluation', flush=True)
start_ollama(CONFIG['ollama_url'], CONFIG['llm_model'], OUT)
TEN, TEN_INDICES = select_ten(TEST, CONFIG['seed'])
LLM_PREDICTIONS = classify_ten(TRAIN, TEN, OUT, CONFIG)
LLM_TABLE = TEN[['paper_id','title','research_domain']].reset_index(drop=True)
for name, pred in LLM_PREDICTIONS.items(): LLM_TABLE[name] = pred
display(LLM_TABLE)
LLM_TABLE.to_csv(OUT/'llm_comparison_10.csv', index=False)
print('DONE: prompts, raw responses and predictions saved incrementally.')

## Comparison tables
Macro precision/recall/F1 weight each domain equally. Full-test ML scores and paired 10-paper scores are separate. **Compare LLM and ML using the paired table.** Ten observations are a small demonstration, not reliable evidence of a model ranking. Corpus, dimension and limited vocabulary also affect embedding comparisons.

In [ ]:
print('START: final comparison tables')
FULL_TABLE = pd.DataFrame(RESULTS).sort_values('F1 (macro)', ascending=False)
display(Markdown('### Full test set: ML systems')); display(FULL_TABLE.round(4))
PAIRED = {name: np.asarray(pred)[TEN_INDICES] for name,pred in PREDICTIONS.items()}
PAIRED.update(LLM_PREDICTIONS)
PAIRED_TABLE = pd.DataFrame([metrics(name, TEN.research_domain, pred, 'Same 10 papers') for name,pred in PAIRED.items()]).sort_values('F1 (macro)', ascending=False)
display(Markdown('### Identical 10 test papers: all systems')); display(PAIRED_TABLE.round(4))
FULL_TABLE.to_csv(OUT/'full_test_results.csv', index=False)
PAIRED_TABLE.to_csv(OUT/'paired_10_results.csv', index=False)
for name,pred in PREDICTIONS.items():
    pd.DataFrame({'System':name, 'paper_id':TEST.paper_id, 'True':TEST.research_domain, 'Predicted':pred}).to_csv(OUT/(name.replace(' ','_').replace('+','plus')+'_predictions.csv'), index=False)
print('DONE: comparison tables and every ML prediction saved.')

## Confusion matrices and misclassified-paper analysis
Rows = true domain, columns = predicted domain. Read the abstracts behind off-diagonal entries; overlapping vocabulary is a possible explanation, not a demonstrated cause. No errors on this templated dataset would not establish real-world generalization.

In [ ]:
print('START: confusion matrices and error inspection')
confusion_plots(PREDICTIONS, TEST, OUT, 'full_test')
confusion_plots(LLM_PREDICTIONS, TEN, OUT, 'llm_10')
ERRORS = pd.concat([misclassified(PREDICTIONS, TEST), misclassified(LLM_PREDICTIONS, TEN)], ignore_index=True)
ERRORS.to_csv(OUT/'misclassified_papers.csv', index=False)
if ERRORS.empty:
    print('No errors among completed systems. This does not prove generalization beyond this dataset.')
else:
    display(ERRORS)
    display(ERRORS.groupby(['System','research_domain','Predicted']).size().rename('Count').reset_index().sort_values('Count', ascending=False))
    display(Markdown('Read each title and abstract above: identify shared terms, ambiguous domains or missing embedding vocabulary before assigning a cause.'))
print('DONE: figures, error titles and full abstracts saved.')

## Simple paper discovery
Change query and optional predicted-domain filter in CONFIG. Retrieval uses cosine similarity in training-fitted BoW space; domain labels below are model predictions.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
print('START: discovery query:', CONFIG['query'])
query_vector = BOW.transform([preprocess_steps(CONFIG['query'])['processed']])
all_vectors = BOW.transform(DF.processed)
found = DF[['paper_id','title']].copy()
found['Similarity'] = cosine_similarity(query_vector, all_vectors).ravel()
found['Predicted domain'] = MODELS['Bag of Words + Logistic Regression'].predict(all_vectors)
if CONFIG['domain_filter']: found = found[found['Predicted domain']==CONFIG['domain_filter']]
display(found.sort_values('Similarity', ascending=False).head(CONFIG['discovery_top_k']))
print('DONE: discovery results.')

## Download results
ZIP includes metrics, plots, split manifests, configuration, environment, prompts, predictions and logs. Save the executed notebook separately for rich outputs. Colab `/content` is temporary; download before disconnecting. Caches allow reuse within the current runtime; this is not training-checkpoint resume.

In [ ]:
import shutil
print('START: package run evidence', OUT)
archive = shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('DONE: results ZIP:', archive)
try:
    from google.colab import files
except ImportError:
    print('Local run ZIP:', archive)
else:
    files.download(archive)